In [1]:
import os
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"

os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [2]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
loader= DirectoryLoader('documents',glob='*.pdf',loader_cls=PyPDFLoader)
docs=loader.lazy_load()

In [4]:
#KG implementation
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class Triple:
    subj: str
    rel: str
    obj: str

class SimpleKG:
    def __init__(self):
        self.triples: List[Triple] = []

    def add_triple(self, subj: str, rel: str, obj: str):
        self.triples.append(Triple(subj, rel, obj))

    def find_triples(self, entity: str) -> List[Triple]:
        # return all triples where entity is subject or object
        return [t for t in self.triples if t.subj == entity or t.obj == entity]


KG = SimpleKG()
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_triples_spacy(text):
    doc = nlp(text)
    triples = []
    for token in doc:
        if token.dep_ in ("ROOT", "relcl"):  # verbs or relations
            subj = [w.text for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w.text for w in token.rights if w.dep_ in ("dobj", "pobj", "attr")]
            if subj and obj:
                triples.append((" ".join(subj), token.lemma_, " ".join(obj)))
    return triples




#link entities in query to KG entities
import spacy
nlp = spacy.load("en_core_web_sm")
def extract_entity_mentions(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents] or [chunk.text for chunk in doc.noun_chunks]

def link_entities(query, kg_entities):
    # simple substring + optional embedding similarity
    mentions = extract_entity_mentions(query)  
    entity_map = {}
    for m in mentions:
        entity_map[m] = [e for e in kg_entities if m.lower() in e.lower()]
    return entity_map

def retrieve_kg_context(query, KG: SimpleKG):
    kg_entities = list(set([t.subj for t in KG.triples] + [t.obj for t in KG.triples]))
    entity_map = link_entities(query, kg_entities)
    triples_text = []
    for ents in entity_map.values():
        for ent in ents:
            for t in KG.find_triples(ent):
                triples_text.append(f"{t.subj} {t.rel} {t.obj}")
    return "\n".join(triples_text)


#combine kg retrieval and context retrieval
def get_combined_context(query, retriever, KG):
    # 1. Retrieve text from FAISS DB
    text_docs = retriever.invoke(query)
    text_context = "\n\n".join([d.page_content for d in text_docs])
    #print(f"context:{text_context}")
    if(len(text_docs)==0):
        return None
    
    # 2. Retrieve KG triples
    kg_context = retrieve_kg_context(query, KG)

    # 3. Combine for final LLM input
    combined_context = f"KG Facts:\n{kg_context}\n\nTextual Context:\n{text_context}"
    return combined_context

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     
    chunk_overlap=100,   
    separators=["\n\n", "\n", " ", ""]
)
chunks = []
for doc in docs:
    triples = extract_triples_spacy(doc.page_content)
    for (subj, rel, obj) in triples:
        KG.add_triple(subj.strip(), rel.strip(), obj.strip())
    chunks.extend(
        text_splitter.split_documents([doc]) 
    )
print(len(chunks))



7


In [6]:
print(chunks[4].page_content[:])

Tax amount = (Retail sale price X tax rate in % of applicable taxes) / (100+ sum of applicable tax rate). 
Explanation. — For the purposes of this rule, — 
(a) “applicable tax” means IGST or CGST or SGST or UTGST as the case may be. 
(b) "retail sale price" means the maximum price declared on goods at which such goods in packaged 
form may be sold to the ultimate consumer and includes all taxes, duties, surcharge or cess by 
whatever name called; 
(c) where on the package of any specified goods more than one retail sale price is declared, the 
maximum of such retail sale price shall be deemed to be the retail sale price; 
(d) where the retail sale price declared on packages of any specified goods is altered to increase the


In [7]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={"device": "cuda:2"},
                                   encode_kwargs={"normalize_embeddings": True}
                                   )

/tmp/ipykernel_3774685/1272004119.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
#FAISS cosine similarity index
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

dimension = 384


index = faiss.IndexFlatIP(dimension)

In [9]:
vectorstore = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore({}), 
    index_to_docstore_id={}
)

vectorstore.add_documents(chunks)
vectorstore.save_local("faiss_index")


In [27]:

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
    )

In [11]:
llm = HuggingFacePipeline.from_model_id(
    # model_id="/mnt/nas/shuvranshu/huggingface_cache/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b", 
    model_id="meta-llama/Llama-3.1-8B",
    # model_id="meta-llama/Llama-3.2-3B-Instruct",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 256,
        "do_sample": False,
    },
    device=1
)


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.31it/s]


In [115]:
# actor_prompt = ChatPromptTemplate.from_messages([
#         ("system", "You are an expert Tax Assistant. Your goal is to provide 100% accurate answers based ONLY on the provided context."),
#     ("user", """
#     ### CURRENT DATA ###
#     USER QUERY: {question}
#     CURRENT CONTEXT: {context}
    
#     ### FEEDBACK FROM AUDITOR ###
#     {feedback_section}
    
#     ### MANDATORY INSTRUCTION ###
#     Based on the 'Required Action' above, you must:
#     1. If 'Refine answer': Correct hallucinations or errors using only the context.
#     2. If 'Re-retrieve context': Acknowledge that current information is insufficient and explain what specific information is missing.
    
#     Answer:
#     """)
#     ])

actor_prompt = PromptTemplate(
    input_variables=["context", "question","feedback_section"],
    template="""
You are a factual assistant. 
feedback from previosu answer: {feedback_section}
based on this critique, understand why the previous answer was low quality.
Use the following context to refine the answer.
Do NOT add information that is not supported by the context.
if you don't find the answer in the context then simply say NO.

Context:
{context}

Question: {question}
Answer:
"""
)

def actor(question, context, feedback_section):
    actor_chain = actor_prompt | llm
    # context = get_combined_context(question,retriever, KG)
    response = actor_chain.invoke({
            "context":context,
            "feedback_section": feedback_section,
            "question":question
            }
            )
    answer=response.split('Answer:')[-1].strip()
    return answer


In [60]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a factual assistant. Use the following context to answer the question.
Do NOT add information that is not supported by the context.
if you don't find the answer in the context then simply say NO.

Context:
{context}

Question: {question}
Answer:
"""
)

In [114]:
question="summarize this document"
context = get_combined_context(question,retriever, KG)
chain = prompt | llm
response = chain.invoke({
        "context":context,
        "question":question,
        }
        )
answer=response.split('Answer:')[-1].strip()
print(f"answer:{answer}")    


answer:Amendment) Rules, 2025. 
(2)   They shall come into force from 1st day of February, 2026. 
2. In the Central Goods and Services Tax Rules, 2017 (hereinafter referred to as the said rules), after rule 31C, 
the following rule shall be inserted, namely: — 
"31D. Value of supply of goods on basis of retail sale price. -(1) Notwithstanding anything contained in 
the provisions of this Chapter, the value of supply of goods bearing the description specified in column (3), 
falling under the corresponding Chapter/ heading/ sub-heading/ tariff item specified in column (2), of the 
Table below, shall be deemed to be the retail sale price declared on such goods, less the amount of tax as 
applicable, namely: -  
Table 
S. No. Chapter / Heading / 
Sub-heading / 
Tariff item 
Description of Goods

Tax amount = (Retail sale price X tax rate in % of applicable taxes) / (100+ sum of applicable tax rate). 
Explanation. — For the purposes of this rule, — 
(a) “applicable tax” means IGST or CGST 

In [80]:
#evaluator
import json
from langchain_community.chat_models import ChatOllama


In [81]:
evaluator_llm= ChatOllama(model="qwen2.5:7b",
                          format="json",
                          temperature=0)
eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a RAG Quality Auditor. You must evaluate ONLY two things:
    1. CONTEXT RELEVANCE: Does the retrieved context contain the information needed to answer the user's query?
    2. FAITHFULNESS: Is the provided answer derived ONLY from the retrieved context?
     
    Output ONLY a JSON object with EXACTLY this structure:
    {{
        "context_relevance": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "faithfulness": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "final_status": "PASS" or "FAIL",
        "action_item": "What should the system do next? (e.g., 'Re-retrieve context' or 'Refine answer')
    }}
    
    Threshold for PASS: Both scores must be > 0.85.
    """),
    ("user", "USER QUERY: {query}\n\nRETRIEVED CONTEXT: {context}\n\nGENERATED ANSWER: {answer}")
])

In [82]:
def evaluate_response(question,context, answer):
    chain = eval_prompt | evaluator_llm
    # response = chain.invoke({"context": context, "answer": answer})
    
    
    try:
        response = chain.invoke({
            "query": question,
            "context": context,
            "answer": answer
        })
        return json.loads(response.content)
    except Exception as e:
        return {
            "final_status": "FAIL",
            "action_item": f"Error in evaluation: {str(e)}"
        }

In [83]:

try:
    response = evaluator_llm.invoke("Hello!")
    print("Ollama is awake and responding!")
except Exception as e:
    print(f"Connection failed. Is the server running? Error: {e}")

Ollama is awake and responding!


In [ ]:
# result = evaluate_response(question,context, answer)
# print(json.dumps(result, indent=2))

{
  "context_relevance": {
    "score": 0.5,
    "reason": "The retrieved context does not directly mention specific tobacco and nicotine products covered by the new valuation rule."
  },
  "faithfulness": {
    "score": 0.0,
    "reason": "The generated answer is not derived from the provided context but rather includes an explanation of the retail sale price."
  },
  "final_status": "FAIL",
  "action_item": "Re-retrieve context"
}


In [84]:
query_refiner_llm = ChatOllama(model="qwen2.5:7b", temperature=0)

def generate_refined_query(original_query, critique):
    prompt = f"""
    The initial search for '{original_query}' failed.
    Critique: {critique}
    based on this critique, write a better search query which contains the missing terms or concepts that the original query lacked,
    which led to the failure.
    Output ONLY the new search string.
    """
    response = query_refiner_llm.invoke(prompt)
    return response.content.strip()

In [123]:
def run_reflexion_loop(original_question,max_iterations=3):
    
    search_query = original_question
    current_context = get_combined_context(search_query, retriever, KG)
    feedback_section = "Required Action:refine answer" 
    
    for i in range(max_iterations):
        print(f"\n--- Iteration {i+1} ---")
        
        answer = actor(original_question, current_context, feedback_section)
        
        evaluation = evaluate_response(original_question, current_context, answer)
        print(json.dumps(evaluation, indent=2))

        if evaluation.get('final_status') == "PASS":
            return answer
        
        action = evaluation.get('action_item', 'Refine answer')
        
        if action == "Re-retrieve context":
            search_query = generate_refined_query(original_question, evaluation['context_relevance']['reason'])
            print(f"new search query:{search_query}")
            current_context = get_combined_context(search_query, retriever, KG)
        else:
            feedback_section=evaluation['faithfulness']['reason']
        # feedback_section = f"""
        # ### FEEDBACK FROM PREVIOUS ATTEMPT ###
        # - Context Relevance Score: {evaluation['context_relevance']['score']}
        # - Context Critique: {evaluation['context_relevance']['reason']}
        
        # - Faithfulness Score: {evaluation['faithfulness']['score']}
        # - Faithfulness Critique: {evaluation['faithfulness']['reason']}
        
        # - Required Action: {evaluation['action_item']}
        # ######################################
        # """
        
        print(f"Critique sent to Actor: {evaluation['action_item']}")
        
    print(" Max iterations reached.")
    return "answer is not present in the document"

In [124]:
question="US tax rules?"
answer=run_reflexion_loop(question)
print(f"answer:{answer}")


--- Iteration 1 ---
{
  "context_relevance": {
    "score": 0.0,
    "reason": "The retrieved context does not contain any information related to US tax rules."
  },
  "faithfulness": {
    "score": 0.0,
    "reason": "The generated answer is not derived from the provided context."
  },
  "final_status": "FAIL",
  "action_item": "Re-retrieve context"
}
new search query:US federal tax rules for individuals and businesses
Critique sent to Actor: Re-retrieve context

--- Iteration 2 ---
{
  "context_relevance": {
    "score": 0.0,
    "reason": "The retrieved context does not contain any information related to US tax rules."
  },
  "faithfulness": {
    "score": 0.0,
    "reason": "The generated answer is not derived from the provided context."
  },
  "final_status": "FAIL",
  "action_item": "Re-retrieve context"
}
new search query:US federal tax rules for individuals and businesses
Critique sent to Actor: Re-retrieve context

--- Iteration 3 ---
{
  "context_relevance": {
    "score": 0

In [ ]:
#Deepeval evaluation

from deepeval.models.base_model import DeepEvalBaseLLM
from langchain_community.chat_models import ChatOllama

class OllamaModel(DeepEvalBaseLLM):

    def __init__(self, model="qwen2.5:7b"):
        self.model_name = model
        self.model = None
        self.load_model()

    def load_model(self):
        self.model = ChatOllama(
            model=self.model_name,
            temperature=0
        )

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        response = await self.model.ainvoke(prompt)
        return response.content

    def get_model_name(self):
        return self.model_name

In [66]:
ollama_model = OllamaModel(model="qwen2.5:7b")

In [67]:
from deepeval.metrics import FaithfulnessMetric, ContextualRelevancyMetric

faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model=ollama_model
)

context_metric = ContextualRelevancyMetric(
    threshold=0.7,
    model=ollama_model
)

In [ ]:
from deepeval.test_case import LLMTestCase


In [ ]:
def evaluation(question,answer,context):
    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[context]
    )

    faithfulness_metric.measure(test_case)
    context_metric.measure(test_case)

    print("Faithfulness:", faithfulness_metric.score)
    print("Context relevance:", context_metric.score)

In [ ]:
def run_reflexion_loop(original_question,max_iterations=5):
    
    search_query = original_question
    current_context = get_combined_context(search_query, retriever, KG)
    feedback_section = "" 
    
    for i in range(max_iterations):
        print(f"\n--- Iteration {i+1} ---")
        
        answer = actor(original_question, current_context, feedback_section)
        
        evaluation = evaluation(original_question, current_context, answer)
        # print(json.dumps(evaluation, indent=2))

        if evaluation.get('final_status') == "PASS":
            return answer
        
        action = evaluation.get('action_item', 'Refine answer')
        
        if action == "Re-retrieve context":
            search_query = generate_refined_query(original_question, evaluation['context_relevance']['reason'])
            print(f"new search query:{search_query}")
            current_context = get_combined_context(search_query, retriever, KG)
        
        feedback_section = f"""
        ### FEEDBACK FROM PREVIOUS ATTEMPT ###
        - Context Relevance Score: {evaluation['context_relevance']['score']}
        - Context Critique: {evaluation['context_relevance']['reason']}
        
        - Faithfulness Score: {evaluation['faithfulness']['score']}
        - Faithfulness Critique: {evaluation['faithfulness']['reason']}
        
        - Required Action: {evaluation['action_item']}
        ######################################
        """
        
        print(f"Critique sent to Actor: {evaluation['action_item']}")
        
    print(" Max iterations reached.")
    return answer

Faithfulness: 0.6666666666666666
Context relevance: 0.4
